In [1]:
import os
from pathlib import Path
from utils import data_loader_utils
from utils import user_defined_functions as udf
import itertools 
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.mlab as mlab

import pandas as pd
import math

from scipy import ndimage
from scipy.stats import pearsonr
import scipy.signal as signal

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

In [2]:
machines = ["M01","M02","M03"]
#process_names = ["OP00","OP01","OP02","OP03","OP04","OP05","OP06","OP07","OP08","OP09","OP10","OP11","OP12","OP13","OP14"]
process_names = ["OP07"]
labels_good = ["good"]
labels_bad = ["bad"]
path_to_dataset = Path("./data/").absolute()

In [3]:
data_good = []
label_data_good = []

for process_name, machine, label in itertools.product(process_names, machines, labels_good):
    data_path = os.path.join(path_to_dataset, machine, process_name, label)
    data_list, data_label = data_loader_utils.load_tool_research_data(data_path, label=label)
    #concatenating
    data_good.extend(data_list)
    label_data_good.extend(data_label)

loading files from c:\Users\20223820\OneDrive - TU Eindhoven\Documents\Courses\Year 4\Q2\Stochastic\5SC29-Group-4-Project\data\M01\OP07\good... 
loading files from c:\Users\20223820\OneDrive - TU Eindhoven\Documents\Courses\Year 4\Q2\Stochastic\5SC29-Group-4-Project\data\M02\OP07\good... 
loading files from c:\Users\20223820\OneDrive - TU Eindhoven\Documents\Courses\Year 4\Q2\Stochastic\5SC29-Group-4-Project\data\M03\OP07\good... 


In [4]:
data_bad = []
label_data_bad = []

for process_name, machine, label in itertools.product(process_names, machines, labels_bad):
    data_path = os.path.join(path_to_dataset, machine, process_name, label)
    data_list, data_label = data_loader_utils.load_tool_research_data(data_path, label=label)
    #concatenating
    data_bad.extend(data_list)
    label_data_bad.extend(data_label)

loading files from c:\Users\20223820\OneDrive - TU Eindhoven\Documents\Courses\Year 4\Q2\Stochastic\5SC29-Group-4-Project\data\M01\OP07\bad... 
loading files from c:\Users\20223820\OneDrive - TU Eindhoven\Documents\Courses\Year 4\Q2\Stochastic\5SC29-Group-4-Project\data\M02\OP07\bad... 
loading files from c:\Users\20223820\OneDrive - TU Eindhoven\Documents\Courses\Year 4\Q2\Stochastic\5SC29-Group-4-Project\data\M03\OP07\bad... 


In [5]:
features_data_tree = []
labels_tree = []
for i in range(len(data_good)):
    features = udf.extract_features(data_good[i][:,0], data_good[i][:,1], data_good[i][:,2])
    features_data_tree.append(features)
    labels_tree.append(1)
for i in range(len(data_bad)):
    features = udf.extract_features(data_bad[i][:,0], data_bad[i][:,1], data_bad[i][:,2])
    features_data_tree.append(features)
    labels_tree.append(0)
features_data_tree = pd.DataFrame(features_data_tree)

In [6]:
y_target_tree = np.array(labels_tree)
X_train, X_test, y_train, y_test = train_test_split(features_data_tree, y_target_tree, test_size=0.3, random_state=42)

In [7]:
rf_model = RandomForestClassifier(
    n_estimators=100, 
    max_depth=5,        # Keep trees shallow to avoid memorizing noise
    random_state=42,
    n_jobs=-1           # Use all CPU cores
)
print("Training Random Forest...")
rf_model.fit(X_train, y_train)

# --- 5. Evaluation ---
preds = rf_model.predict(X_test)
acc = accuracy_score(y_test, preds)

print("-" * 30)
print(f"Random Forest Accuracy: {acc * 100:.2f}%")
print("-" * 30)

print("Feature Importances:")
# This tells you which physical characteristic actually flagged the fault
importances = pd.Series(rf_model.feature_importances_, index=features_data_tree.columns).sort_values(ascending=False)
print(importances)

print("\nConfusion Matrix (True Neg, False Pos, False Neg, True Pos):")
print(confusion_matrix(y_test, preds))

Training Random Forest...
------------------------------
Random Forest Accuracy: 100.00%
------------------------------
Feature Importances:
x_psd_max                 0.226756
z_psd_max                 0.210000
x_psd_rms                 0.190000
y_psd_max                 0.169197
y_psd_rms                 0.109188
z_psd_rms                 0.056934
max_cross_corr_xy         0.030000
x_psd_kurtosis            0.007925
cross_corr_xy_kurtosis    0.000000
y_psd_kurtosis            0.000000
z_psd_kurtosis            0.000000
dtype: float64

Confusion Matrix (True Neg, False Pos, False Neg, True Pos):
[[ 4  0]
 [ 0 44]]
